# E6R dataset producer — the depth arm of the E6 ablation, flat encoder, reps = 2

E6 measured the flat-encoded single-rep representation on four
fixed-degree-sequence strata and found the collapse; E6D measured the
degree-encoded single-rep arm and did not restore decodability. E6R varies
the remaining circuit knob: repetitions. The circuit is encode once, then
(entangle, mix) twice — exactly `reps = 2` in the census class — with the
**flat encoder** (Luke's call for this arm; the degree-encoded reps-2 arm
is deferred). Everything else is held fixed: the **identical 1,600
graphs**, the same targets, the same entangler and mixer angles per
block.

**Pre-registration.** No directional expectation is registered. The
second (entangle, mix) block is not a rescaling of the first (the second
entangler acts on the mixed state, so the composition is genuinely deeper
interference rather than a doubled angle), and whether depth restores any
off-regularity decodability is exactly the open question. The flat E6
and degree E6D results stand as delivered and are not touched.

**Pipeline discipline.** The circuit class is the E6 flat-encoder variant
of the census `QuIK_circuit` (the one-line flat encoder Luke approved for
E6), constructed with `reps = 2`, simulated through qiskit `Statevector`.
No custom simulation code appears anywhere in this notebook.

**Provenance discipline.** This producer does not sample graphs. It loads
the flat E6 pkl, re-asserts its integrity, copies the canonical graph6
strings and every graph-level target verbatim, and replaces exactly one
field per entry: `exact_vector`. One graph per stratum is printed with
its L1 distance to the reps-1 flat vector to confirm the depth change
took effect.

**Runtime.** Roughly 1,600 statevectors at n = 14 with two entangler
blocks, about three minutes.

## Environment

In [ ]:
!pip install 'qiskit[visualization]' --quiet

In [ ]:
import pickle

import numpy as np
import networkx as nx

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

from tqdm.notebook import tqdm

In [ ]:
SEED = 0
NUM_VERTICES = 14

# Input: the flat-encoder reps-1 E6 dataset this arm re-embeds.
FLAT_E6_BASE = "/kaggle/input/notebooks/lukemiller1987/aaai-n14-e6-dataset-producer/"

EXPECTED_STRATA = ('S1_near_regular', 'S2_bimodal', 'S3_skewed', 'S4_hub')
EXPECTED_GRAPHS_PER_STRATUM = 400

# Circuit parameters — canonical angles; the varied knob is reps.
FLAT_ENCODING_ANGLE = 2.875
ENTANGLER_SCALAR = 2.0
MIXER_SCALAR = 0.1
REPS = 2

# Fields copied verbatim from the flat pkl (graph invariants of identical graphs).
COPIED_FIELDS = ('graph6', 'adj_mat', 'C3', 'C4', 'C5', 'C6', 'girth',
                 'diameter', 'radius', 'wiener', 'spectrum', 'spectral_gap')

## Load the flat E6 dataset and re-assert its integrity

In [ ]:
with open(FLAT_E6_BASE + "n14_e6_data_dict.pkl", 'rb') as flat_file:
    flat_pkl = pickle.load(flat_file)
flat_data = flat_pkl['data']
flat_meta = flat_pkl['meta']

assert flat_meta['circuit']['encoding'].startswith('flat'), (
    'input pkl is not the flat-encoder E6 run')
assert flat_meta['circuit']['reps'] == 1, 'input pkl is not the reps-1 run'
assert tuple(sorted(flat_data)) == tuple(sorted(EXPECTED_STRATA))

for stratum_name in EXPECTED_STRATA:
    graph_indices = sorted(flat_data[stratum_name])
    assert len(graph_indices) == EXPECTED_GRAPHS_PER_STRATUM, (
        f'{stratum_name}: expected {EXPECTED_GRAPHS_PER_STRATUM} graphs, '
        f'got {len(graph_indices)}')

    expected_degree_sequence = sorted(flat_meta['sequences'][stratum_name])
    canonical_strings = [flat_data[stratum_name][graph_index]['graph6']
                         for graph_index in graph_indices]
    assert len(set(canonical_strings)) == len(canonical_strings), (
        f'{stratum_name}: duplicate canonical graph6 strings')

    for graph_index in graph_indices:
        entry = flat_data[stratum_name][graph_index]
        adjacency_matrix = entry['adj_mat'].astype(np.int64)
        degree_sequence = adjacency_matrix.sum(axis=1)
        assert sorted(degree_sequence.tolist()) == expected_degree_sequence, (
            f'{stratum_name} graph {graph_index}: degree sequence broken')

        adjacency_cubed = adjacency_matrix @ adjacency_matrix @ adjacency_matrix
        assert np.trace(adjacency_cubed) == 6 * entry['C3'], (
            f'{stratum_name} graph {graph_index}: tr(A^3) identity broken')
        adjacency_fourth = np.linalg.matrix_power(adjacency_matrix, 4)
        degree_squares_sum = (degree_sequence ** 2).sum()
        assert (np.trace(adjacency_fourth)
                == 8 * entry['C4'] + 2 * degree_squares_sum
                - degree_sequence.sum()), (
            f'{stratum_name} graph {graph_index}: tr(A^4) identity broken')

    print(f'{stratum_name}: {len(graph_indices)} graphs loaded, '
          f'identities exact, canonical strings distinct')

## The circuit — the E6 flat-encoder variant, reps = 2

The class is the census `QuIK_circuit` with the one-line flat encoder
approved for E6 (uniform RX(2.875) on every qubit; identical to the
degree encoder on regular graphs, verified bit-identically in E6).
`reps = 2` appends the (entangler, mixer) block twice, which is the
class's existing repetition mechanism, untouched.

In [ ]:
class QuIK_circuit:
    """Builds the QuIC circuit for a given adjacency matrix.

    Flat RX encoder (the E6 variant), RZZ entangler over edges, uniform RX
    mixer, (entangler + mixer) repeated `reps` times. Construction only.

    Census class with the E6 flat-encoder line; reps is the E6R knob.
    """

    def __init__(self, adj_mat,
                 flat_encoding_angle=FLAT_ENCODING_ANGLE,
                 entangler_scalar=ENTANGLER_SCALAR,
                 mixer_scalar=MIXER_SCALAR,
                 reps=REPS):
        self.adj_mat = np.asarray(adj_mat)
        self.flat_encoding_angle = flat_encoding_angle
        self.entangler_scalar = entangler_scalar
        self.mixer_scalar = mixer_scalar
        self.reps = reps

        self.num_qubits = self.adj_mat.shape[0]
        self.qubits = list(range(self.num_qubits))

        self.quik_circuit = self.build_quik_circuit()

    def build_encoder(self):
        encoder = QuantumCircuit(self.num_qubits)
        encoder.rx(self.flat_encoding_angle, self.qubits, 'Flat Encoder')
        return encoder

    def build_entangler(self):
        entangler = QuantumCircuit(self.num_qubits)
        edge_u, edge_v = np.nonzero(np.triu(self.adj_mat, k=1))
        for u, v in zip(edge_u, edge_v):
            entangler.rzz(self.entangler_scalar, u, v)
        return entangler

    def build_mixer(self):
        mixer = QuantumCircuit(self.num_qubits)
        mixer.rx(self.mixer_scalar, self.qubits, 'Mixer')
        return mixer

    def build_quik_circuit(self):
        self.encoder = self.build_encoder()
        self.mixer = self.build_mixer()
        self.entangler = self.build_entangler()

        circuit = self.encoder.copy()
        for _ in range(self.reps):
            circuit.append(self.entangler, self.qubits)
            circuit.append(self.mixer, self.qubits)
        circuit.measure_all()
        return circuit


def sorted_exact_vector(adjacency_matrix):
    """Exact sorted output distribution through the qiskit pipeline at
    reps = 2, mirroring the census producers line for line."""
    circuit = QuIK_circuit(adjacency_matrix).quik_circuit.decompose()
    circuit = circuit.remove_final_measurements(inplace=False)
    born_probabilities = Statevector.from_instruction(circuit).probabilities()
    return np.sort(born_probabilities)[::-1]

## Re-embed every graph at depth two

In [ ]:
depth_two_data = {}
for stratum_name in EXPECTED_STRATA:
    depth_two_data[stratum_name] = {}
    graph_indices = sorted(flat_data[stratum_name])

    for graph_index in tqdm(graph_indices, desc=f'{stratum_name} embedding'):
        flat_entry = flat_data[stratum_name][graph_index]
        new_entry = {field: flat_entry[field] for field in COPIED_FIELDS}

        depth_two_vector = sorted_exact_vector(flat_entry['adj_mat'])
        normalization_error = abs(depth_two_vector.sum() - 1.0)
        assert normalization_error < 1e-12, (
            f'{stratum_name} graph {graph_index}: '
            f'normalization error {normalization_error:.2e}')
        assert np.all(np.diff(depth_two_vector) <= 0), (
            f'{stratum_name} graph {graph_index}: vector not sorted descending')

        new_entry['exact_vector'] = depth_two_vector
        depth_two_data[stratum_name][graph_index] = new_entry

    first_index = graph_indices[0]
    l1_depth_two_vs_flat = np.abs(
        depth_two_data[stratum_name][first_index]['exact_vector']
        - flat_data[stratum_name][first_index]['exact_vector']).sum()
    print(f'{stratum_name}: graph {first_index} '
          f'L1(reps2, reps1) = {l1_depth_two_vs_flat:.3e} '
          f'(nonzero confirms the depth change took effect)')

## Persist

In [ ]:
readme_text = """QuIC EXACT EMBEDDINGS — E6R DEPTH ARM (FLAT ENCODER, REPS=2), n=14
================================================================

Generated: 2026-07-16
Producer:  AAAI_n14_E6R_dataset_producer (Kaggle)
Author:    Luke Miller (UMKC)
Context:   E6R of the AAAI 2027 QuIC characterization study — the
           repetition arm of the E6 ablation, sanctioned 2026-07-16
           after the degree-encoded arm (E6D) did not restore
           off-regularity decodability. Flat encoder per Luke; the
           degree-encoded reps-2 arm is deferred. The flat E6 and
           degree E6D results stand unchanged.

CONTENTS
--------
n14_e6r_data_dict.pkl
    dict keyed by stratum name -> graph index (0..399). The graphs,
    canonical graph6 strings, and all graph-level targets are copied
    VERBATIM from n14_e6_data_dict.pkl (the flat reps-1 run). The
    only field this producer computes is exact_vector.

    Circuit: flat RX(2.875) encoder on every qubit, then
    (RZZ(2.0) per edge, RX(0.1) mixer) repeated TWICE — encode,
    entangle, mix, entangle, mix. Simulated through the qiskit
    Statevector pipeline (the census class with the E6 flat-encoder
    line, reps=2). Circuit objects are not stored.

    Identities tr(A^3) = 6*C3 and tr(A^4) = 8*C4 + 2*sum(d^2) -
    sum(d) re-asserted on every graph at load.
"""

with open('/kaggle/working/n14_e6r_readme.txt', 'w') as readme_file:
    readme_file.write(readme_text)

output_pkl = {
    'data': depth_two_data,
    'meta': {
        'seed': SEED,
        'n_vertices': NUM_VERTICES,
        'sequences': flat_meta['sequences'],
        'source': 'graphs and targets copied verbatim from n14_e6_data_dict.pkl '
                  '(flat reps-1 run); exact_vector recomputed at reps=2',
        'circuit': {'flat_encoding_angle': FLAT_ENCODING_ANGLE,
                    'entangler_scalar': ENTANGLER_SCALAR,
                    'mixer_scalar': MIXER_SCALAR, 'reps': REPS,
                    'encoding': 'flat — uniform RX on every qubit; '
                                '(entangler, mixer) block repeated twice'},
    },
}
with open('/kaggle/working/n14_e6r_data_dict.pkl', 'wb') as output_file:
    pickle.dump(output_pkl, output_file)
print('saved n14_e6r_data_dict.pkl and n14_e6r_readme.txt')

## Results

(Written after the run. The reading order: (1) the load gates and
identity asserts; (2) the L1(reps2, reps1) lines, which must be far from
zero; (3) nothing else — this producer makes no comparative claims. The
comparison belongs to the E6R consumer, which reuses the flat run's
frozen splits and prints reps-1 versus reps-2 side by side.)